# TinyChirp CNN-Time TensorFlow

Train a 1D CNN on raw audio waveforms (no spectrogram), export an int8 TFLite model, and write a Rust `audio_sample.rs` file.

This mirrors `building_tensorflow/tinychirp_cnn.ipynb` but replaces the 2D mel CNN with a 1D time CNN similar to `StreamingCNNArch` from the TinyChirp `CNN_Time` model.

In [1]:
from typing import TYPE_CHECKING
from pathlib import Path
import random
import gc
import os

import numpy as np
import tensorflow as tf

if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

tf.config.optimizer.set_jit(False)

# Also, limit GPU memory growth to prevent the 'factory registration' errors
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
os.environ['TF_XLA_FLAGS'] = '--tf_xla_auto_jit=0'
# Disable the oneDNN floating point noise
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
# Prevent TF from grabbing all VRAM
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

import tensorflow as tf
# Verify it's off
print(f"XLA Status: {tf.config.optimizer.get_jit()}")

# Reproducibility
SEED = 3407
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Paths
REPO_ROOT = Path.cwd().parent
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR = REPO_ROOT / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT = REPO_ROOT / "dataset"

OUT_TFLITE = MODELS_DIR / "cnn_time_tf.tflite"
OUT_AUDIO_RS = SRC_DIR / "audio_sample.rs"

# Audio geometry (must match Rust)
SAMPLE_RATE = 16000
FRAME_LENGTH = 1024
FRAME_STEP = 256
TARGET_FRAMES = 184
TARGET_AUDIO_LEN = (TARGET_FRAMES - 1) * FRAME_STEP + FRAME_LENGTH

print("Dataset root:", DATASET_ROOT)
print("Model output:", OUT_TFLITE)
print("Audio sample output:", OUT_AUDIO_RS)

2026-04-01 11:09:16.471162: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-01 11:09:16.473227: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-01 11:09:16.475969: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-01 11:09:16.482891: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-01 11:09:16.496522: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registe

XLA Status: 
Dataset root: /home/nathan/Documents/tiny-chirp-microflow/dataset
Model output: /home/nathan/Documents/tiny-chirp-microflow/models/cnn_time_tf.tflite
Audio sample output: /home/nathan/Documents/tiny-chirp-microflow/src/audio_sample.rs


2026-04-01 11:09:18.328156: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-04-01 11:09:18.339426: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2251] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [2]:
def fix_audio_length(audio):
    # audio: [batch, time, 1] from keras audio_dataset_from_directory
    audio = tf.squeeze(audio, axis=-1)  # [batch, time]
    audio = audio[:, :TARGET_AUDIO_LEN]
    current_len = tf.shape(audio)[1]
    pad_len = tf.maximum(0, TARGET_AUDIO_LEN - current_len)
    audio = tf.pad(audio, [[0, 0], [0, pad_len]])
    audio = tf.ensure_shape(audio, [None, TARGET_AUDIO_LEN])
    audio = tf.expand_dims(audio, axis=-1)  # [batch, time, 1]
    return audio


def to_features(audio, label):
    audio = fix_audio_length(audio)
    return audio, label


train_ds_raw = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "training",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=32,
    shuffle=True,
    seed=3407,
)
val_ds_raw = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "validation",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=32,
    shuffle=False,
)
test_ds_raw = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "testing",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=32,
    shuffle=False,
)
label_names = np.array(train_ds_raw.class_names)
num_labels = len(label_names)
print("Classes:", label_names)
train_ds = train_ds_raw.map(to_features, num_parallel_calls=tf.data.AUTOTUNE).prefetch(2).repeat()
val_ds = val_ds_raw.map(to_features, num_parallel_calls=tf.data.AUTOTUNE).prefetch(2).repeat()
test_ds = test_ds_raw.map(to_features, num_parallel_calls=tf.data.AUTOTUNE).prefetch(2).repeat()

Found 11292 files belonging to 2 classes.


2026-04-01 11:09:18.543273: I tensorflow_io/core/kernels/cpu_check.cc:128] Your CPU supports instructions that this TensorFlow IO binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA


Found 1380 files belonging to 2 classes.
Found 1393 files belonging to 2 classes.
Classes: ['non_target' 'target']


In [3]:
model = keras.Sequential([
    keras.layers.Input(shape=(TARGET_AUDIO_LEN, 1)),
    keras.layers.Conv1D(4, 3, activation="relu"),
    keras.layers.MaxPooling1D(pool_size=2, strides=2),
    keras.layers.Conv1D(8, 3, activation="relu"),
    keras.layers.GlobalAveragePooling1D(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(num_labels),
])
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
history = model.fit(train_ds, validation_data=val_ds, epochs=2, steps_per_epoch=len(train_ds_raw), validation_steps=len(val_ds_raw))
print("Test metrics:", model.evaluate(test_ds, verbose=0, steps=len(test_ds_raw)))

Epoch 1/2
353/353 ━━━━━━━━━━━━━━━━━━━━ 25s 69ms/step - accuracy: 0.6819 - loss: 0.5774 - val_accuracy: 0.7420 - val_loss: 0.4560
Epoch 2/2
  4/353 ━━━━━━━━━━━━━━━━━━━━ 23s 68ms/step - accuracy: 0.7500 - loss: 0.4418

2026-04-01 11:09:44.538022: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 12255488 bytes after encountering the first element of size 6127744 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


353/353 ━━━━━━━━━━━━━━━━━━━━ 28s 78ms/step - accuracy: 0.8077 - loss: 0.4078 - val_accuracy: 0.8080 - val_loss: 0.3486
Test metrics: [0.3831583857536316, 0.8205312490463257]


In [4]:
# 1. Gather representative data
val_specs = []
for x_batch, _ in test_ds.unbatch().take(100):
    sample = x_batch.numpy().astype(np.float32)
    sample = np.reshape(sample, (1, TARGET_AUDIO_LEN, 1))
    val_specs.append(sample)

def representative_data_gen():
    for sample in val_specs:
        yield [sample]

# 2. WORKAROUND FOR KERNEL CRASH: Save to disk first
# This prevents Keras 3 / MLIR C++ segfaults during INT8 shape inference
model.export("temp_saved_model") # If on older TF, use model.save("temp_saved_model")
converter = tf.lite.TFLiteConverter.from_saved_model("temp_saved_model")

# 3. Setup Converter
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# NOTE: Removed the experimental flag and the tf.device context manager!

# 4. Convert and Save
try:
    tflite_bytes = converter.convert()
    OUT_TFLITE.write_bytes(tflite_bytes)
    print(f"Success! Wrote {OUT_TFLITE}")
except Exception as e:
    print(f"Conversion failed: {e}")

INFO:tensorflow:Assets written to: temp_saved_model/assets


2026-04-01 11:10:14.731430: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
INFO:tensorflow:Assets written to: temp_saved_model/assets


Saved artifact at 'temp_saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 47872, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  126071862296096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071862294336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071857758800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071857759680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071862295744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071862295920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071857759504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  126071857760032: TensorSpec(shape=(), dtype=tf.resource, name=None)


W0000 00:00:1775034614.955684   35801 tf_tfl_flatbuffer_helpers.cc:390] Ignored output_format.
W0000 00:00:1775034614.955706   35801 tf_tfl_flatbuffer_helpers.cc:393] Ignored drop_control_dependency.
2026-04-01 11:10:14.955988: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: temp_saved_model
2026-04-01 11:10:14.956463: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-04-01 11:10:14.956472: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: temp_saved_model
2026-04-01 11:10:14.961958: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:388] MLIR V1 optimization pass is not enabled
2026-04-01 11:10:14.962681: I tensorflow/cc/saved_model/loader.cc:234] Restoring SavedModel bundle.
2026-04-01 11:10:14.986757: I tensorflow/cc/saved_model/loader.cc:218] Running initialization op on SavedModel bundle at path: temp_saved_model
2026-04-01 11:10:14.993857: I tensorflow/cc/saved_model/loader.cc

Success! Wrote /home/nathan/Documents/tiny-chirp-microflow/models/cnn_time_tf.tflite


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8


In [5]:
raw_sample_ds = keras.utils.audio_dataset_from_directory(
    DATASET_ROOT / "testing",
    labels="inferred",
    sampling_rate=SAMPLE_RATE,
    batch_size=1,
    shuffle=False,
)

clips_by_label = {0: [], 1: []}
for audio_batch, label_batch in raw_sample_ds.unbatch():
    label = int(label_batch.numpy())
    if len(clips_by_label[label]) < 2:
        fixed = fix_audio_length(tf.expand_dims(audio_batch, 0))[0].numpy().astype(np.float32)
        clips_by_label[label].append(fixed)
    if len(clips_by_label[0]) == 2 and len(clips_by_label[1]) == 2:
        break

ordered = [
    ("non_target", clips_by_label[0][0]),
    ("target", clips_by_label[1][0]),
    ("non_target", clips_by_label[0][1]),
    ("target", clips_by_label[1][1]),
]

rs = []
rs.append("// Generated by building_tensorflow/cnn_time.ipynb\n")
rs.append(f"pub const SAMPLE_RATE: usize = {SAMPLE_RATE};\n\n")
rs.append("pub struct TestClip {\n")
rs.append("    pub expected_label: &'static str,\n")
rs.append("    pub source_file: &'static str,\n")
rs.append("    pub audio: &'static [f32],\n")
rs.append("}\n\n")

for i, (_label, audio) in enumerate(ordered, 1):
    audio_vals = ", ".join(f"{float(v):.8f}" for v in audio)
    rs.append(f"pub const CLIP_{i}: &[f32] = &[{audio_vals}];\n\n")

rs.append("pub const TEST_CLIPS: [TestClip; 4] = [\n")
for i, (label, _audio) in enumerate(ordered, 1):
    rs.append("    TestClip {\n")
    rs.append(f"        expected_label: \"{label}\",\n")
    rs.append(f"        source_file: \"dataset/testing/{label}/sample_{i}.wav\",\n")
    rs.append(f"        audio: CLIP_{i},\n")
    rs.append("    },\n")
rs.append("];\n")

OUT_AUDIO_RS.write_text("".join(rs), encoding="utf-8")
print("Wrote", OUT_AUDIO_RS, "clips=", len(ordered), "samples_per_clip=", len(ordered[0][1]))

Found 1393 files belonging to 2 classes.


Wrote /home/nathan/Documents/tiny-chirp-microflow/src/audio_sample.rs clips= 4 samples_per_clip= 47872


/tmp/ipykernel_35801/1471352182.py:35: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  audio_vals = ", ".join(f"{float(v):.8f}" for v in audio)
